# Quick Quantum QSVC Model Training
Minimal notebook to train and save QSVC model cleanly

In [2]:
import numpy as np
import pandas as pd
import pickle
import dill
import json
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from qiskit_aer import AerSimulator
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

In [3]:
project_root = Path.cwd().parent
phase3_dir = project_root / 'phase3'
phase4_dir = project_root / 'phase4'
phase4_dir.mkdir(exist_ok=True)
(phase4_dir / 'models').mkdir(exist_ok=True)

domain_pca_path = phase3_dir / 'domain' / 'domain_pca.csv'
domain_pca_df = pd.read_csv(domain_pca_path)

X_domain_full = domain_pca_df.drop('label', axis=1).values
y_domain = domain_pca_df['label'].values

print(f"Loaded data shape: {X_domain_full.shape}")
print(f"Label distribution: {np.bincount(y_domain.astype(int))}")

Loaded data shape: (199944, 4)
Label distribution: [ 99944 100000]


In [4]:
n_qubits = 4
X_domain_pca = X_domain_full[:, :n_qubits]

X_train, X_test, y_train, y_test = train_test_split(
    X_domain_pca, y_domain, test_size=0.2, random_state=42, stratify=y_domain
)

train_size = min(500, len(X_train))
X_train_q = X_train[:train_size]
y_train_q = y_train[:train_size]
X_test_q = X_test[:min(150, len(X_test))]
y_test_q = y_test[:min(150, len(X_test))]

scaler = StandardScaler()
X_train_q_scaled = scaler.fit_transform(X_train_q)
X_test_q_scaled = scaler.transform(X_test_q)

In [5]:
feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2, entanglement='full')
backend = AerSimulator()
qk = FidelityQuantumKernel(feature_map=feature_map)

print("Training QSVC...")
start_time = time.time()
qsvc = QSVC(quantum_kernel=qk)
qsvc.fit(X_train_q_scaled, y_train_q)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f}s")

Training QSVC...
Training completed in 811.65s


In [6]:
y_pred = qsvc.predict(X_test_q_scaled)

acc = accuracy_score(y_test_q, y_pred)
prec = precision_score(y_test_q, y_pred, zero_division=0)
rec = recall_score(y_test_q, y_pred, zero_division=0)
f1 = f1_score(y_test_q, y_pred, zero_division=0)
cm = confusion_matrix(y_test_q, y_pred)

print("QSVC RESULTS:")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"\nConfusion Matrix:\n{cm}")

QSVC RESULTS:
Accuracy:  0.7400 (74.00%)
Precision: 0.7073
Recall:    0.7945
F1 Score:  0.7484

Confusion Matrix:
[[53 24]
 [15 58]]


In [7]:
qsvc_model_path = phase4_dir / 'models' / 'qsvc_domain_model.dill'
scaler_path = phase4_dir / 'models' / 'quantum_scaler.pkl'

with open(qsvc_model_path, 'wb') as f:
    dill.dump(qsvc, f)

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

metadata = {
    'model_type': 'QSVC',
    'n_qubits': n_qubits,
    'feature_map': 'ZZFeatureMap',
    'feature_map_reps': 2,
    'n_training_samples': len(X_train_q_scaled),
    'n_test_samples': len(X_test_q_scaled),
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1_score': float(f1),
    'training_time_seconds': float(training_time)
}

metadata_path = phase4_dir / 'models' / 'qsvc_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nModels saved:")
print(f"  QSVC: {qsvc_model_path}")
print(f"  Scaler: {scaler_path}")
print(f"  Metadata: {metadata_path}")


Models saved:
  QSVC: d:\final_year_project\Qunatum-Enhanced_Threat_Intelligence\phase4\models\qsvc_domain_model.dill
  Scaler: d:\final_year_project\Qunatum-Enhanced_Threat_Intelligence\phase4\models\quantum_scaler.pkl
  Metadata: d:\final_year_project\Qunatum-Enhanced_Threat_Intelligence\phase4\models\qsvc_metadata.json
